[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyTorch and TensorFlow, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)

# Training Loops


## What you will be able to do

Write a PyTorch training loop that trains in batches and measures the validation stations after
every epoch, get the same from Keras's `fit`, and read the record of losses to see when more
training stopped helping.

## The idea

### The problem

The loops so far printed the loss on the stations they trained on, and measured the test stations
only once training had finished. The training loss can keep falling while the model gets no better,
or gets worse, at stations it has never seen, because a model can lower it by fitting the quirks of
those particular 800 stations. A planned station is one it has never seen, so training needs a
second measure, taken on stations the steps never use.

That is what the validation stations are for. The test stations stay untouched until the end, for
one final measure of the model you chose. With batches, epochs, a pass over the validation
stations, a switch between training and evaluation, and a record of every epoch's losses, a loop
has enough parts to be worth writing once, carefully, and reusing.

### A formal definition

> A **training loop** repeats **epochs**. In each one it puts the model in training mode and, for
> every batch, computes the loss and its gradients and takes an optimizer step. Then it puts the
> model in evaluation mode and computes the loss on a **validation set**, examples held out from
> training, without recording gradients. The **test set** is held out too, and is used once, at
> the end.

### Why it works that way

`model.train()` and `model.eval()` set a flag that some layers read. Dropout, which the
**Overfitting and Regularization** notebook adds, switches off parts of the network at random in
training and never in evaluation. A model without such layers gives the same answers either way,
and the loop switches anyway, so that adding one later cannot quietly spoil the validation losses.

The validation pass runs inside `torch.no_grad()`. Nothing computed there needs a gradient, and
without a record of the operations PyTorch keeps less in memory. An epoch's loss is the average over
its batches weighted by their sizes, so that the short last batch counts for what it holds.

Keras's `fit` is the same loop. Given `validation_data`, it computes the validation loss after
every epoch, and the `History` object it returns holds every epoch's losses in a dictionary.
Choosing the epoch, or the model, by its test loss would make the test loss part of the choice, and
it would then flatter the model it chose, which is why the choosing is left to the validation loss.

### Where this shows up

The **Overfitting and Regularization** notebook uses the validation loss to stop training early, and
the **A Complete Project** notebook trains the snow classifier from start to finish with a validation
pass, measuring the test stations once at the end.

### What this notebook covers

- a training pass and a validation pass as two small functions
- the loop over epochs, with a record of every epoch's losses
- what `model.train()` and `model.eval()` change
- Keras's `fit` with validation data, its `History`, and a training step written with `GradientTape`
- the snow classifier trained both ways, with accuracy
- three mistakes, two of which raise nothing

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import torch

torch.manual_seed(0)
x = torch.linspace(-1, 1, 20).unsqueeze(1)
y = 3 * x + 1
model = torch.nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
for epoch in range(3):
    model.train()
    optimizer.zero_grad()
    loss = torch.nn.functional.mse_loss(model(x[::2]), y[::2])
    loss.backward()
    optimizer.step()
    model.eval()
    with torch.no_grad():
        valid_loss = torch.nn.functional.mse_loss(model(x[1::2]), y[1::2])
    print(epoch, round(loss.item(), 3), round(valid_loss.item(), 3))
```

```
0 3.4 1.421
1 1.357 0.575
2 0.549 0.232
```

Twenty points on a line, every other one for training and the rest for validation. Each epoch takes
one step on the training points, then measures the points it did not train on, and both losses fall.

## Setup

Three libraries, PyTorch's data tools, a way to quiet TensorFlow while it loads, and the stations.

- `numpy` builds the stations, and finds the best epoch in a record of losses
- `torch` is PyTorch, and `DataLoader` and `TensorDataset` serve its batches
- `tensorflow` is TensorFlow, and `keras` its library of layers and training tools
- `contextlib` and `os` hide the messages TensorFlow prints while it loads
- `io` and `re` catch and tidy what Keras prints, used once, in Common errors

TensorFlow's C++ code prints several lines as it loads, each stamped with the time and a process
number, so no two runs would show the same output. Setting `TF_CPP_MIN_LOG_LEVEL` to 3 hides what
TensorFlow logs once it has started, and `c_messages_hidden` sends the lines it prints while it
starts to `os.devnull`. The cell prints PyTorch's version without the part after `+`, which names
the build.

`station_network_arrays` makes the 1,200 made-up weather stations of the **scikit-learn, Deep Dive**
guide, drawn from a fixed seed, so they are the same on every run. `features` holds each station's
latitude, longitude, elevation and distance from the coast as 32-bit floats, `jan_mean` its January
mean in degrees Celsius, and `snow_on_may_1` a 1 for a station with snow on 1 May and a 0 otherwise.
The cell splits the stations into 800 for training, 200 for validation and 200 for testing, and
standardizes the features with the training stations' means into `scaled`. This notebook uses all
three sets.

In [1]:
import contextlib
import io
import os
import re

import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset


@contextlib.contextmanager
def c_messages_hidden():
    """Send what C and C++ code writes to the error stream to os.devnull while the block runs."""
    saved = os.dup(2)
    devnull = os.open(os.devnull, os.O_WRONLY)
    os.dup2(devnull, 2)
    try:
        yield
    finally:
        os.dup2(saved, 2)
        os.close(devnull)
        os.close(saved)


os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
with c_messages_hidden():
    import tensorflow as tf
    import keras


def station_network_arrays(n=1200, seed=2025):
    """The made-up stations of the scikit-learn guide, as arrays: four features and two targets."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    shift = np.where(coast_km < 3, 0.8,
            np.where(elevation > 700, -0.5,
            np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), 0.3, 0.0)))
    rng.choice(4, n, p=[0.4, 0.3, 0.1, 0.2])   # each station's owner, which this guide leaves out
    rng.integers(1950, 2025, n)                 # and the year it was installed
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    features = np.column_stack([latitude.round(2), longitude.round(2), elevation.round(0), coast_km.round(1)])
    return features.astype("float32"), jan_mean.round(1).astype("float32"), (may_mean < 2.5).astype("float32")


FEATURES = ["latitude", "longitude", "elevation_m", "coast_km"]
features, jan_mean, snow_on_may_1 = station_network_arrays()
order = np.random.default_rng(0).permutation(len(features))
train_rows, valid_rows, test_rows = order[:800], order[800:1000], order[1000:]
mean, std = features[train_rows].mean(axis=0), features[train_rows].std(axis=0)
scaled = (features - mean) / std

print("torch", torch.__version__.split("+")[0], "| tensorflow", tf.__version__, "| keras", keras.__version__,
      "|", len(features), "stations")

torch 2.11.0 | tensorflow 2.21.0 | keras 3.13.2 | 1200 stations


## Worked examples

### Training, validation and test stations

`as_tensors` turns one set of rows into features and a target in a column. The training batches are
shuffled, and the validation stations are read in one batch of 200, in order, since their order
changes nothing.

In [2]:
def as_tensors(rows, target):
    """The standardized features of these rows, and their target as a column."""
    return torch.from_numpy(scaled[rows]), torch.from_numpy(target[rows]).unsqueeze(1)


X_train, y_train = as_tensors(train_rows, jan_mean)
X_valid, y_valid = as_tensors(valid_rows, jan_mean)
X_test, y_test = as_tensors(test_rows, jan_mean)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True,
                          generator=torch.Generator().manual_seed(0))
valid_loader = DataLoader(TensorDataset(X_valid, y_valid), batch_size=200)
print(len(train_loader), "training batches |", len(valid_loader), "validation batch")

13 training batches | 1 validation batch


### A training pass and a validation pass

`train_one_epoch` takes a step on every batch and returns the epoch's average loss. `evaluate`
measures a loader's examples in evaluation mode, with nothing recorded.

In [3]:
def train_one_epoch(model, loader, optimizer, loss_fn):
    """One pass over the loader's batches, with a step for each; returns the average loss."""
    model.train()
    total, count = 0.0, 0
    for xb, yb in loader:
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(xb)
        count += len(xb)
    return total / count


def evaluate(model, loader, loss_fn):
    """The average loss over the loader's examples, in evaluation mode, with no gradients."""
    model.eval()
    total, count = 0.0, 0
    with torch.no_grad():
        for xb, yb in loader:
            total += loss_fn(model(xb), yb).item() * len(xb)
            count += len(xb)
    return total / count

### The loop over epochs and its record

The cell trains the 4-32-1 network of the **Why PyTorch and TensorFlow** notebook with Adam for 40
epochs.
`history` keeps both losses for every epoch, and the cell prints every fifth.

In [4]:
torch.manual_seed(0)
net = torch.nn.Sequential(torch.nn.Linear(4, 32), torch.nn.ReLU(), torch.nn.Linear(32, 1))
optimizer = torch.optim.Adam(net.parameters(), lr=0.01)
mse = torch.nn.MSELoss()

history = {"loss": [], "val_loss": []}
for epoch in range(1, 41):
    history["loss"].append(train_one_epoch(net, train_loader, optimizer, mse))
    history["val_loss"].append(evaluate(net, valid_loader, mse))
    if epoch % 5 == 0:
        print(f"epoch {epoch:2}: training loss {history['loss'][-1]:.3f}, validation loss {history['val_loss'][-1]:.3f}")

best_epoch = int(np.argmin(history["val_loss"])) + 1
print("lowest validation loss at epoch", best_epoch, f"({min(history['val_loss']):.3f})")

epoch  5: training loss 2.226, validation loss 2.098
epoch 10: training loss 1.150, validation loss 1.122
epoch 15: training loss 0.970, validation loss 0.950
epoch 20: training loss 0.905, validation loss 0.867
epoch 25: training loss 0.851, validation loss 0.807
epoch 30: training loss 0.819, validation loss 0.786
epoch 35: training loss 0.813, validation loss 0.777
epoch 40: training loss 0.833, validation loss 0.786
lowest validation loss at epoch 33 (0.775)


Both losses fall quickly at first and level off near 0.8. The validation loss is lowest at epoch 33,
0.775, and slightly higher by epoch 40, 0.786, so the last few epochs gained nothing on stations the
model has not trained on.

In [5]:
print(f"test loss after 40 epochs: {evaluate(net, DataLoader(TensorDataset(X_test, y_test), batch_size=200), mse):.3f}")

test loss after 40 epochs: 0.975


The test loss, 0.975, is measured once, after the loop has finished. The test stations are a
different sample from the validation stations, so the two losses differ. The **Overfitting and
Regularization** notebook keeps the weights from the best epoch, where this loop keeps the last.

### What train and eval switch

Every module has a `training` flag, which `train()` and `eval()` set for it and for every layer
inside it. This network has no layer that reads the flag, so its answers are the same in both modes.

In [6]:
net.train()
print("after train():", net.training, net[0].training)
with torch.no_grad():
    in_training_mode = net(X_valid)
net.eval()
print("after eval(): ", net.training, net[0].training)
with torch.no_grad():
    in_evaluation_mode = net(X_valid)
print("the same answers:", torch.equal(in_training_mode, in_evaluation_mode))

after train(): True True
after eval():  False False
the same answers: True


`eval()` cleared the flag on the network and on its first layer, and the answers are identical. A
network with a dropout layer gives different answers in the two modes, which the **Overfitting and
Regularization** notebook shows.

### Keras's fit with validation data

`validation_data` gives `fit` the validation stations, and the `History` it returns holds a list per
measure, one value per epoch.

In [7]:
keras.utils.set_random_seed(0)
net_keras = keras.Sequential([keras.Input((4,)), keras.layers.Dense(32, activation="relu"), keras.layers.Dense(1)])
net_keras.compile(optimizer=keras.optimizers.Adam(0.01), loss="mse")
fitted = net_keras.fit(scaled[train_rows], jan_mean[train_rows],
                       validation_data=(scaled[valid_rows], jan_mean[valid_rows]),
                       epochs=40, batch_size=64, verbose=0)

print(list(fitted.history), len(fitted.epoch), "epochs")
for epoch in range(5, 41, 5):
    print(f"epoch {epoch:2}: training loss {fitted.history['loss'][epoch - 1]:.3f}, "
          f"validation loss {fitted.history['val_loss'][epoch - 1]:.3f}")

['loss', 'val_loss'] 40 epochs
epoch  5: training loss 2.137, validation loss 1.834
epoch 10: training loss 1.083, validation loss 1.026
epoch 15: training loss 0.916, validation loss 0.892
epoch 20: training loss 0.865, validation loss 0.831
epoch 25: training loss 0.835, validation loss 0.798
epoch 30: training loss 0.827, validation loss 0.792
epoch 35: training loss 0.821, validation loss 0.784
epoch 40: training loss 0.812, validation loss 0.769


Keras's network follows the same course as PyTorch's and ends at a validation loss of 0.769, against
0.786, from different starting weights. `fitted.epoch` counts from 0, so epoch 5 is at position 4 of
each list. `evaluate` and `predict` take `verbose=0`, as `fit` does.

In [8]:
print(f"test loss: {net_keras.evaluate(scaled[test_rows], jan_mean[test_rows], verbose=0):.3f}")
predicted = net_keras.predict(scaled[test_rows][:3], verbose=0)
print("predicted:", predicted.shape, predicted.ravel().round(1), "| measured:", jan_mean[test_rows][:3])

test loss: 0.996
predicted: (3, 1) [-5.  -5.4 -4.6] | measured: [-6.1 -5.2 -3.2]


The test loss is 0.996. `predict` returns a NumPy array with a row per station, and the first three
test stations are predicted within 1.4 degrees of their measured means.

### A training step of your own in Keras

`fit` hides its loop. Written out with `GradientTape`, a Keras loop has the same parts as the
PyTorch one. `training=True` is Keras's version of `model.train()`, passed on each call.

In [9]:
keras.utils.set_random_seed(0)
net_tape = keras.Sequential([keras.Input((4,)), keras.layers.Dense(32, activation="relu"), keras.layers.Dense(1)])
optimizer_keras = keras.optimizers.Adam(0.01)
mse_keras = keras.losses.MeanSquaredError()
batches = tf.data.Dataset.from_tensor_slices((scaled[train_rows], jan_mean[train_rows][:, None])).shuffle(800, seed=0).batch(64)

for epoch in range(40):
    for xb, yb in batches:
        with tf.GradientTape() as tape:
            loss = mse_keras(yb, net_tape(xb, training=True))
        gradients = tape.gradient(loss, net_tape.trainable_variables)
        optimizer_keras.apply_gradients(zip(gradients, net_tape.trainable_variables))

valid_loss = mse_keras(jan_mean[valid_rows][:, None], net_tape(scaled[valid_rows], training=False))
print(f"validation loss after 40 epochs: {float(valid_loss):.3f}")

validation loss after 40 epochs: 0.786


The validation loss after 40 epochs is 0.786, close to `fit`'s 0.769, since the network, the
optimizer and the batch size are the same, and only the shuffling differs.

### The snow classifier in both frameworks

The same loop trains a classifier when it is given `BCEWithLogitsLoss` and the snow labels.
`accuracy` counts a logit above 0 as a prediction of snow.

In [10]:
snow_train = as_tensors(train_rows, snow_on_may_1)
snow_valid = as_tensors(valid_rows, snow_on_may_1)
snow_test = as_tensors(test_rows, snow_on_may_1)
snow_loader = DataLoader(TensorDataset(*snow_train), batch_size=64, shuffle=True,
                         generator=torch.Generator().manual_seed(0))
snow_valid_loader = DataLoader(TensorDataset(*snow_valid), batch_size=200)


def accuracy(model, X, y):
    """The share of examples whose logit is on the right side of 0."""
    model.eval()
    with torch.no_grad():
        return ((model(X) > 0).float() == y).float().mean().item()


torch.manual_seed(0)
classifier = torch.nn.Sequential(torch.nn.Linear(4, 32), torch.nn.ReLU(), torch.nn.Linear(32, 1))
optimizer = torch.optim.Adam(classifier.parameters(), lr=0.01)
bce = torch.nn.BCEWithLogitsLoss()
for epoch in range(1, 41):
    train_loss = train_one_epoch(classifier, snow_loader, optimizer, bce)
    if epoch % 10 == 0:
        print(f"epoch {epoch}: training loss {train_loss:.3f}, validation loss "
              f"{evaluate(classifier, snow_valid_loader, bce):.3f}, validation accuracy "
              f"{accuracy(classifier, *snow_valid):.3f}")
print(f"test accuracy: {accuracy(classifier, *snow_test):.3f}")

epoch 10: training loss 0.315, validation loss 0.260, validation accuracy 0.875
epoch 20: training loss 0.309, validation loss 0.253, validation accuracy 0.875
epoch 30: training loss 0.305, validation loss 0.255, validation accuracy 0.865
epoch 40: training loss 0.306, validation loss 0.255, validation accuracy 0.865
test accuracy: 0.820


The validation accuracy is 0.875 at epoch 10 and 0.865 by epoch 40, and the test accuracy is 0.82.

In [11]:
keras.utils.set_random_seed(0)
classifier_keras = keras.Sequential([keras.Input((4,)), keras.layers.Dense(32, activation="relu"), keras.layers.Dense(1)])
classifier_keras.compile(optimizer=keras.optimizers.Adam(0.01),
                         loss=keras.losses.BinaryCrossentropy(from_logits=True),
                         metrics=[keras.metrics.BinaryAccuracy(threshold=0.0)])
fitted_snow = classifier_keras.fit(scaled[train_rows], snow_on_may_1[train_rows],
                                   validation_data=(scaled[valid_rows], snow_on_may_1[valid_rows]),
                                   epochs=40, batch_size=64, verbose=0)

print(list(fitted_snow.history))
print(f"validation accuracy after 40 epochs: {fitted_snow.history['val_binary_accuracy'][-1]:.3f}")
test_loss, test_accuracy = classifier_keras.evaluate(scaled[test_rows], snow_on_may_1[test_rows], verbose=0)
print(f"test accuracy: {test_accuracy:.3f}")

['binary_accuracy', 'loss', 'val_binary_accuracy', 'val_loss']
validation accuracy after 40 epochs: 0.865
test accuracy: 0.820


Keras's classifier ends at the same validation accuracy, 0.865, and the same test accuracy, 0.82.
Given `metrics`, the history gains a list for each metric, and `val_` before a name marks the value on
the validation stations.

## Your turn

Four tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/10-training-loops-solutions.ipynb).

**1.** Train a fresh 4-32-1 network from `torch.manual_seed(0)` with `torch.optim.SGD` at a learning rate of 0.05 for 40 epochs, using `train_one_epoch` and `evaluate`, and print the epoch with the lowest validation loss.

In [12]:
# your code here

**2.** Fit the Keras network again from `keras.utils.set_random_seed(0)` with `batch_size=32`, and print the validation loss at epochs 10, 20, 30 and 40.

In [13]:
# your code here

**3.** Find the epoch with the highest validation accuracy in `fitted_snow.history`, with `np.argmax`, and print it with its accuracy.

In [14]:
# your code here

**4.** Train a three-band classifier with the PyTorch loop, where the bands are `np.digitize(jan_mean, [-8.0, -3.0])` as in the **Losses and Optimizers** notebook, a 4-32-3 network and `CrossEntropyLoss`, for 40 epochs, and print its validation accuracy.

In [15]:
# your code here

## Common errors

### No error, and a loss that never settles

This is the loop of the worked examples with one line taken out, `optimizer.zero_grad()`.

In [16]:
def train_one_epoch_without_zero_grad(model, loader, optimizer, loss_fn):
    """train_one_epoch, missing its zero_grad()."""
    model.train()
    total, count = 0.0, 0
    for xb, yb in loader:
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(xb)
        count += len(xb)
    return total / count


loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True,
                    generator=torch.Generator().manual_seed(0))
torch.manual_seed(0)
unzeroed = torch.nn.Sequential(torch.nn.Linear(4, 32), torch.nn.ReLU(), torch.nn.Linear(32, 1))
optimizer = torch.optim.Adam(unzeroed.parameters(), lr=0.01)
for epoch in range(1, 41):
    train_loss = train_one_epoch_without_zero_grad(unzeroed, loader, optimizer, mse)
    if epoch % 5 == 0:
        print(f"epoch {epoch:2}: training loss {train_loss:.3f}, validation loss {evaluate(unzeroed, valid_loader, mse):.3f}")

epoch  5: training loss 17.822, validation loss 6.023
epoch 10: training loss 22.346, validation loss 18.249
epoch 15: training loss 6.688, validation loss 11.534
epoch 20: training loss 70.801, validation loss 27.776
epoch 25: training loss 14.363, validation loss 19.508
epoch 30: training loss 51.628, validation loss 28.078
epoch 35: training loss 39.261, validation loss 42.056
epoch 40: training loss 293.365, validation loss 300.946


`backward()` adds each batch's gradients to whatever `.grad` holds, so without `zero_grad()`
every step uses the sum of every gradient since the start, and a step no longer follows the batch it
was computed on. Nothing raises an error, and the losses jump around at many times the 0.8 the
same network reached in the worked example, so call `optimizer.zero_grad()` before every
`backward()`.

### `KeyError: 'val_loss'`

The cell runs one more epoch of `fit`, given no validation data.

In [17]:
one_more = net_keras.fit(scaled[train_rows], jan_mean[train_rows], epochs=1, batch_size=64, verbose=0)
print(list(one_more.history))
one_more.history["val_loss"]

['loss']


KeyError: 'val_loss'

A `History` holds only what `fit` measured. Without `validation_data`, or `validation_split`, there
is no validation loss, so the dictionary has no `val_loss` key. Pass the validation stations to every
`fit` whose history you read, or read `one_more.history.get("val_loss")`, which returns `None`.

### No error, and a progress bar from predict

The cell calls `predict` and `evaluate` without `verbose=0`. `last_progress_line` catches what a call prints and
returns its last line, without its colors, and with the timings and the loss replaced by `…`, since
they can change from run to run.

In [18]:
def last_progress_line(call):
    """Run call, catch what it prints, and return its last line with the timings and loss replaced by …."""
    printed = io.StringIO()
    with contextlib.redirect_stdout(printed):
        call()
    text = re.sub(r"\x1b\[[0-9;]*m", "", printed.getvalue())
    line = text.split("\r")[-1].split("\b")[-1].strip()
    return re.sub(r"loss: [\d.e+-]+", "loss: …", re.sub(r"\d+(ms|us|s)/step", "…/step", re.sub(r"\b\d+s\b", "…", line)))


print(last_progress_line(lambda: net_keras.predict(scaled[valid_rows])))
print(last_progress_line(lambda: net_keras.evaluate(scaled[valid_rows], jan_mean[valid_rows])))

7/7 ━━━━━━━━━━━━━━━━━━━━ … …/step
7/7 ━━━━━━━━━━━━━━━━━━━━ … …/step - loss: …


By default `predict` draws a progress bar of its batches, 7 of them for 200 stations, with the
seconds taken and the time per step, and `evaluate` draws the same with the loss beside it. In a
notebook that is output you did not ask for, and it changes on every run. Pass `verbose=0` to both,
as the worked examples do for `fit` too.

## Recap

- You can write a training pass and a validation pass, and loop over epochs with a record of both
  losses.
- You can switch a model between training and evaluation, and measure without recording gradients.
- You can get the same from Keras's `fit` with `validation_data`, and read its `History`.
- You can write a Keras training step with `GradientTape`, and train a classifier with either loop.

## What is next

The **Overfitting and Regularization** notebook trains a larger network on fewer stations, watches
the validation loss turn, and stops training at its best epoch.

---

&#8592; **Previous:** [Batches and Data Pipelines](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/09-batches-and-data-pipelines.ipynb)  &nbsp;·&nbsp;  [PyTorch and TensorFlow, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Overfitting and Regularization](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/11-overfitting-and-regularization.ipynb) &#8594;
